In [1]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, explained_variance_score, f1_score, log_loss, matthews_corrcoef, max_error, mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, mean_squared_log_error, median_absolute_error, precision_score, r2_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

from eda import (
    build_categorical_count_figures,
    build_numeric_histogram_figures,
    build_numeric_summary,
    build_scatter_vs_target_figures,
    build_shapiro_summary,
    plot_correlation_heatmap,
    plot_erros_tecnicos_poisson,
    plot_missing_values,
    plot_target_distribution,
    plot_transparency_proportion,
)
from models import (
    build_binary_target,
    train_decision_tree_regressor,
    train_linear_regression,
    train_neural_network_regressor,
    train_logistic_regression_classifier,
    train_random_forest_regressor,
)
from pre_processing import transform_dataset


In [2]:
ROOT = Path('.')
DATA_PATH = ROOT / 'Dataset-iGov.csv'
ARTIFACTS_DIR = ROOT / 'artifacts'
EDA_ARTIFACTS_DIR = ARTIFACTS_DIR / 'eda'
PREPROCESSING_ARTIFACTS_DIR = ARTIFACTS_DIR / 'preprocessing'
MLRUNS_DIR = ROOT / 'mlruns'
TARGET_COLUMN = 'indicador_kpi'
CLASSIFICATION_THRESHOLD = 70.0
RANDOM_STATE = 42

for directory in [ARTIFACTS_DIR, EDA_ARTIFACTS_DIR, PREPROCESSING_ARTIFACTS_DIR, MLRUNS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(MLRUNS_DIR.resolve().as_uri())
mlflow.set_experiment('tp1c-igov-pipeline')


c:\Users\Ricardo\miniconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Traceback (most recent call last):
  File "c:\Users\Ricardo\miniconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Ricardo\miniconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File

<Experiment: artifact_location='file:///C:/Users/Ricardo/Documents/GitHub/AOOP/Tp1C/mlruns/941245034444576193', creation_time=1774353035346, experiment_id='941245034444576193', last_update_time=1774353035346, lifecycle_stage='active', name='tp1c-igov-pipeline', tags={}, workspace='default'>

In [3]:
def sanitize_name(name: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_-]+', '_', name).strip('_').lower()


def log_dataframe_artifact(df: pd.DataFrame, output_path: Path, artifact_path: str) -> None:
    df.to_csv(output_path, index=False)
    mlflow.log_artifact(str(output_path), artifact_path=artifact_path)


def log_figure(fig, artifact_file: str) -> None:
    mlflow.log_figure(fig, artifact_file)
    plt.close(fig)


def log_figure_dict(figures: dict, artifact_dir: str) -> None:
    for name, fig in figures.items():
        file_name = sanitize_name(name)
        log_figure(fig, f'{artifact_dir}/{file_name}.png')


In [4]:
def run_eda_and_log(df: pd.DataFrame) -> None:
    numeric_summary = build_numeric_summary(df).reset_index().rename(columns={'index': 'feature'})
    shapiro_summary = build_shapiro_summary(df)

    log_dataframe_artifact(numeric_summary, EDA_ARTIFACTS_DIR / 'numeric_summary.csv', 'eda/tables')
    log_dataframe_artifact(shapiro_summary, EDA_ARTIFACTS_DIR / 'shapiro_summary.csv', 'eda/tables')

    log_figure(plot_missing_values(df), 'eda/missing_values.png')
    log_figure(plot_target_distribution(df), 'eda/target_distribution.png')
    log_figure(plot_transparency_proportion(df), 'eda/transparency_proportion.png')
    log_figure(plot_erros_tecnicos_poisson(df), 'eda/erros_tecnicos_poisson.png')
    log_figure(plot_correlation_heatmap(df), 'eda/correlation_heatmap.png')

    log_figure_dict(build_numeric_histogram_figures(df), 'eda/histograms')
    log_figure_dict(build_scatter_vs_target_figures(df), 'eda/scatter_vs_target')
    log_figure_dict(build_categorical_count_figures(df), 'eda/categorical_counts')


In [5]:
def safe_mape(y_true, y_pred) -> float:
    non_zero_mask = y_true != 0
    if non_zero_mask.sum() == 0:
        return float('nan')
    return mean_absolute_percentage_error(y_true[non_zero_mask], y_pred[non_zero_mask])


def safe_msle(y_true, y_pred) -> float:
    if (y_true < 0).any() or (y_pred < 0).any():
        return float('nan')
    return mean_squared_log_error(y_true, y_pred)


def adjusted_r2_score(y_true, y_pred, n_features: int) -> float:
    n_samples = len(y_true)
    if n_samples <= n_features + 1:
        return float('nan')
    r2 = r2_score(y_true, y_pred)
    return 1 - ((1 - r2) * (n_samples - 1) / (n_samples - n_features - 1))


def plot_regression_diagnostics(y_true, y_pred, model_name: str):
    residuals = y_true - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].scatter(y_true, y_pred, alpha=0.7, edgecolors='k')
    min_value = min(y_true.min(), y_pred.min())
    max_value = max(y_true.max(), y_pred.max())
    axes[0].plot([min_value, max_value], [min_value, max_value], color='red', linestyle='--')
    axes[0].set_title(f'{model_name}: Actual vs Predicted')
    axes[0].set_xlabel('Actual')
    axes[0].set_ylabel('Predicted')
    axes[0].grid(alpha=0.3)

    axes[1].hist(residuals, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
    axes[1].set_title(f'{model_name}: Residual Distribution')
    axes[1].set_xlabel('Residual')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    return fig


def plot_confusion_matrix_figure(y_true, y_pred, model_name: str):
    matrix = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(f'{model_name}: Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    fig.tight_layout()
    return fig


def train_and_log_regressor(run_name: str, trainer, X_train, X_test, y_train, y_test, model_params: dict) -> None:
    with mlflow.start_run(run_name=run_name, nested=True):
        mlflow.log_params(model_params)
        model = trainer(X_train, y_train, **model_params)
        predictions = model.predict(X_test)

        metrics = {
            'mse': mean_squared_error(y_test, predictions),
            'mae': mean_absolute_error(y_test, predictions),
            'rmse': mean_squared_error(y_test, predictions) ** 0.5,
            'r2': r2_score(y_test, predictions),
            'adjusted_r2': adjusted_r2_score(y_test, predictions, X_test.shape[1]),
            'explained_variance': explained_variance_score(y_test, predictions),
            'median_absolute_error': median_absolute_error(y_test, predictions),
            'max_error': max_error(y_test, predictions),
            'mape': safe_mape(y_test, predictions),
            'msle': safe_msle(y_test, predictions),
            'mean_prediction': float(predictions.mean()),
            'prediction_std': float(predictions.std()),
        }
        mlflow.log_metrics(metrics)
        log_figure(plot_regression_diagnostics(y_test, predictions, run_name), f'metrics/{run_name}_regression_diagnostics.png')
        mlflow.sklearn.log_model(model, artifact_path='model')


def train_and_log_logistic(run_name: str, X_train, X_test, y_train, y_test, model_params: dict) -> None:
    with mlflow.start_run(run_name=run_name, nested=True):
        mlflow.log_params(model_params)
        mlflow.log_param('classification_threshold', CLASSIFICATION_THRESHOLD)

        model = train_logistic_regression_classifier(X_train, y_train, **model_params)
        predictions = model.predict(X_test)
        probabilities = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

        metrics = {
            'accuracy': accuracy_score(y_test, predictions),
            'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
            'precision': precision_score(y_test, predictions, zero_division=0),
            'recall': recall_score(y_test, predictions, zero_division=0),
            'f1': f1_score(y_test, predictions, zero_division=0),
            'matthews_corrcoef': matthews_corrcoef(y_test, predictions),
            'cohen_kappa': cohen_kappa_score(y_test, predictions),
            'positive_rate_predicted': float(predictions.mean()),
            'positive_rate_actual': float(y_test.mean()),
        }

        if probabilities is not None:
            metrics['roc_auc'] = roc_auc_score(y_test, probabilities)
            metrics['average_precision'] = average_precision_score(y_test, probabilities)
            metrics['log_loss'] = log_loss(y_test, probabilities)

        mlflow.log_metrics(metrics)

        report = classification_report(y_test, predictions, output_dict=True, zero_division=0)
        report_df = pd.DataFrame(report).transpose().reset_index().rename(columns={'index': 'label'})
        log_dataframe_artifact(report_df, ARTIFACTS_DIR / f'{run_name}_classification_report.csv', 'metrics')
        log_figure(plot_confusion_matrix_figure(y_test, predictions, run_name), f'metrics/{run_name}_confusion_matrix.png')
        mlflow.sklearn.log_model(model, artifact_path='model')


In [7]:
with mlflow.start_run(run_name='full_pipeline'):
    mlflow.log_param('dataset_path', str(DATA_PATH))
    mlflow.log_param('target_column', TARGET_COLUMN)
    mlflow.log_param('classification_threshold', CLASSIFICATION_THRESHOLD)

    raw_df = pd.read_csv(DATA_PATH)
    mlflow.log_metric('row_count', float(len(raw_df)))
    mlflow.log_metric('column_count', float(raw_df.shape[1]))

    run_eda_and_log(raw_df)

    processed_df, processed_dataset_path, preprocessor_path = transform_dataset(
        raw_df,
        output_dir=PREPROCESSING_ARTIFACTS_DIR,
        target_column=TARGET_COLUMN,
    )

    mlflow.log_artifact(str(processed_dataset_path), artifact_path='preprocessing')
    mlflow.log_artifact(str(preprocessor_path), artifact_path='preprocessing')

    X = processed_df.drop(columns=[TARGET_COLUMN])
    y_regression = processed_df[TARGET_COLUMN]
    y_classification = build_binary_target(y_regression, threshold=CLASSIFICATION_THRESHOLD)

    X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
        X,
        y_regression,
        test_size=0.2,
        random_state=RANDOM_STATE,
    )

    X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
        X,
        y_classification,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_classification,
    )

    train_and_log_regressor(
        run_name='linear_regression',
        trainer=train_linear_regression,
        X_train=X_train_reg,
        X_test=X_test_reg,
        y_train=y_train_reg,
        y_test=y_test_reg,
        model_params={},
    )

    train_and_log_regressor(
        run_name='decision_tree_regressor',
        trainer=train_decision_tree_regressor,
        X_train=X_train_reg,
        X_test=X_test_reg,
        y_train=y_train_reg,
        y_test=y_test_reg,
        model_params={'random_state': RANDOM_STATE, 'max_depth': 6},
    )

    train_and_log_regressor(
        run_name='random_forest_regressor',
        trainer=train_random_forest_regressor,
        X_train=X_train_reg,
        X_test=X_test_reg,
        y_train=y_train_reg,
        y_test=y_test_reg,
        model_params={'random_state': RANDOM_STATE, 'n_estimators': 200},
    )

    train_and_log_regressor(
        run_name='neural_network_regressor',
        trainer=train_neural_network_regressor,
        X_train=X_train_reg,
        X_test=X_test_reg,
        y_train=y_train_reg,
        y_test=y_test_reg,
        model_params={
            'random_state': RANDOM_STATE,
            'hidden_layer_sizes': (32, 16),
            'activation': 'relu',
            'solver': 'adam',
            'early_stopping': True,
            'max_iter': 1000,
        },
    )

    train_and_log_logistic(
        run_name='logistic_regression_classifier',
        X_train=X_train_clf,
        X_test=X_test_clf,
        y_train=y_train_clf,
        y_test=y_test_clf,
        model_params={'random_state': RANDOM_STATE, 'max_iter': 1000},
    )

print('MLflow pipeline completed successfully.')


Null values per column:
id_registo                0
data_registo              0
unidade_organizacional    0
tipo_servico              0
indicador_si              0
taxa_resolucao            0
tempo_resposta            0
satisfacao_cidadao        0
volume_interacoes         0
canal_utilizado           0
taxa_abandono             0
erros_tecnicos            0
transparencia             0
feedback_cidadao          0
segmentacao_utilizador    0
area_tematica             0
indicador_kpi             0
Numeric columns normalized: ['indicador_si', 'taxa_resolucao', 'tempo_resposta', 'satisfacao_cidadao', 'volume_interacoes', 'taxa_abandono', 'erros_tecnicos', 'registo_ano', 'registo_mes', 'registo_dia']
Categorical columns encoded: ['unidade_organizacional', 'tipo_servico', 'canal_utilizado', 'transparencia', 'feedback_cidadao', 'segmentacao_utilizador', 'area_tematica']
Processed dataset saved to: artifacts\preprocessing\processed_dataset.csv
Preprocessor saved to: artifacts\preprocessing\prep

2026/03/28 21:02:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 21:02:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/28 21:02:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 21:02:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

MLflow pipeline completed successfully.
